In [4]:
from qiskit import QuantumCircuit, transpile
import random

# Generate benchmark circuits: Clifford-only, Clifford+T, and random Clifford+T-like circuits

def create_random_circuit(n, g, t_count, h_count, seed=None):
    """Create a random circuit on n qubits with exactly g gates, at least t_count T gates and h_count H gates."""
    if g < (t_count + h_count):
        raise ValueError('g must be >= t_count + h_count')
    rnd = random.Random(seed)
    circ = QuantumCircuit(n, name=f"random_{n}q_g{g}_t{t_count}_h{h_count}")
    gates = []
    # enforce required T and H counts
    gates.extend(['t'] * t_count)
    gates.extend(['h'] * h_count)
    # fill remaining gates from a Clifford+T gate set
    remaining = g - t_count - h_count
    choices = ['s', 'cx', 'h']
    for _ in range(remaining):
        gates.append(rnd.choice(choices))
    rnd.shuffle(gates)
    for gate in gates:
        if gate == 'cx':
            if n < 2:
                q = rnd.randrange(n)
                circ.h(q)
            else:
                a, b = rnd.sample(range(n), 2)
                circ.cx(a, b)
        elif gate == 't':
            q = rnd.randrange(n)
            circ.t(q)
        elif gate == 'h':
            q = rnd.randrange(n)
            circ.h(q)
        elif gate == 's':
            q = rnd.randrange(n)
            circ.s(q)
    circ.metadata = { 'kind': f"random_g{g}_t{t_count}_h{h_count}" }
    return circ

def create_benchmark_clifford_circuits():
    circuits = []
    num_qubits_list = [2,3,11,21]
    for n in num_qubits_list:
        # Clifford-only circuit (H, S, CX)
        clifford = QuantumCircuit(n, name=f"clifford_{n}q")
        for i in range(n):
            clifford.h(i)
        for i in range(n-1):
            clifford.cx(i, i+1)
        for i in range(n):
            clifford.s(i)
        clifford.metadata = {"kind": "clifford"}
        circuits.append(clifford)

        # Clifford+T circuit (add T gates)
        clifford_t = QuantumCircuit(n, name=f"clifford_t_{n}q")
        for i in range(n):
            clifford_t.h(i)
        for i in range(n-1):
            clifford_t.cx(i, i+1)
        for i in range(n):
            clifford_t.s(i)
            clifford_t.t(i)
        clifford_t.metadata = {"kind": "clifford_t"}
        circuits.append(clifford_t)

        # Add a true random circuit with specified gate/T/H counts
        # Example defaults: g = max(10, n*5), t_count = max(1, n), h_count = max(1, n*2)
        try:
            random_circ = create_random_circuit(n, g=max(10, n*5), t_count=max(1, n), h_count=max(1, n*2))
            circuits.append(random_circ)
        except ValueError as e:
            # skip if parameters invalid for this n
            print(f"Skipping random circuit for {n} qubits: {e}")

    # Transpile to Clifford+T basis to standardize gates
    transpiled = [transpile(circ, basis_gates=["cx", "z", "h", "s", "t"], optimization_level=1) for circ in circuits]
    for i, circ in enumerate(transpiled):
        circ.metadata = circuits[i].metadata
        circ.name = circuits[i].name
    return transpiled

circuits = create_benchmark_clifford_circuits()
# for idx, qc in enumerate(circuits):
    # display(qc.draw('mpl'))

In [5]:
# Export transpiled scalable circuits to QASM2 files
import os
import qiskit.qasm2

export_dir = "qasm2_exports"
os.makedirs(export_dir, exist_ok=True)

for idx, qc in enumerate(circuits):
    qasm_str = qiskit.qasm2.dumps(qc)
    # Use circuit metadata to distinguish kinds (e.g., 'clifford', 'clifford_t')
    kind = getattr(qc, 'metadata', {}).get('kind', 'unknown')
    # sanitize kind for filenames
    kind_safe = str(kind).replace(' ', '_')
    filename = os.path.join(export_dir, f"{kind_safe}_{qc.num_qubits}q_{idx+1}.qasm2")
    with open(filename, "w") as f:
        f.write(qasm_str)
    print(f"Exported {filename}")

Exported qasm2_exports/clifford_2q_1.qasm2
Exported qasm2_exports/clifford_t_2q_2.qasm2
Exported qasm2_exports/random_g10_t2_h4_2q_3.qasm2
Exported qasm2_exports/clifford_3q_4.qasm2
Exported qasm2_exports/clifford_t_3q_5.qasm2
Exported qasm2_exports/random_g15_t3_h6_3q_6.qasm2
Exported qasm2_exports/clifford_11q_7.qasm2
Exported qasm2_exports/clifford_t_11q_8.qasm2
Exported qasm2_exports/random_g55_t11_h22_11q_9.qasm2
Exported qasm2_exports/clifford_21q_10.qasm2
Exported qasm2_exports/clifford_t_21q_11.qasm2
Exported qasm2_exports/random_g105_t21_h42_21q_12.qasm2


In [2]:
# Simulate each circuit to get the statevector and benchmark Qiskit
import time
from qiskit_aer import Aer
from qiskit import transpile

simulator = Aer.get_backend('statevector_simulator')

statevectors = []
timings = []

for idx, qc in enumerate(circuits):
    start = time.time()
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc).result()
    statevector = result.get_statevector()
    elapsed = time.time() - start
    statevectors.append(statevector)
    timings.append(elapsed)
    print(f"Simulated circuit {idx+1} ({qc.num_qubits} qubits) in {elapsed:.4f} seconds")

Simulated circuit 1 (2 qubits) in 0.0612 seconds
Simulated circuit 2 (2 qubits) in 0.0479 seconds
Simulated circuit 3 (2 qubits) in 0.0474 seconds
Simulated circuit 4 (3 qubits) in 0.0474 seconds
Simulated circuit 5 (3 qubits) in 0.0472 seconds
Simulated circuit 6 (3 qubits) in 0.0508 seconds
Simulated circuit 7 (11 qubits) in 0.0495 seconds
Simulated circuit 8 (11 qubits) in 0.0481 seconds
Simulated circuit 9 (11 qubits) in 0.0484 seconds
Simulated circuit 10 (21 qubits) in 0.1244 seconds
Simulated circuit 11 (21 qubits) in 0.1784 seconds
Simulated circuit 12 (21 qubits) in 0.4345 seconds


## Benchmarking Clifford+T Circuits and Exporting to QASM2
This notebook generates multiple Clifford+T circuits using Qiskit and exports them to QASM2 format for benchmarking.

In [3]:
# Cleanup exported QASM files once the benchmark is done
from pathlib import Path

export_dir = Path("qasm2_exports")
if not export_dir.exists():
    print("No exported QASM files to clean up.")
else:
    removed = []
    for qasm_path in export_dir.glob("*.qasm2"):
        qasm_path.unlink()
        removed.append(qasm_path.name)
    print(f"Removed {len(removed)} exported QASM files: {removed}")

Removed 8 exported QASM files: ['clifford_3q_3.qasm2', 'clifford_2q_1.qasm2', 'clifford_21q_7.qasm2', 'clifford_t_11q_6.qasm2', 'clifford_t_3q_4.qasm2', 'clifford_t_21q_8.qasm2', 'clifford_t_2q_2.qasm2', 'clifford_11q_5.qasm2']
